In [ ]:
%run init_notebook.py
import json
import math
import os
import queue
import threading

import torch
import torchaudio.transforms as T
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.dataset import NSynth, nsynth_collate_fn
from src.diffusion import *
from src.load_encoders import load_VAE, load_cVAE
from src.paths import *
from src.utils.models import (
    adjust_shape,
    compute_magnitude_and_phase,
    compute_magnitude_and_phase_sin_cos,
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract latents from VAE and cVAE

functions:

In [ ]:
def extract_latents_VAE(model, dataset, stft_transform, save_dir, variational=True, batch_size=256, save_queue_size=8):
    os.makedirs(save_dir, exist_ok=True)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    save_q = queue.Queue(maxsize=save_queue_size)

    def saver_worker():
        while True:
            item = save_q.get()
            if item is None:
                save_q.task_done()
                break
            tensor, path = item
            torch.save(tensor, path)
            save_q.task_done()

    saver_thread = threading.Thread(target=saver_worker, daemon=True)
    saver_thread.start()

    pbar = tqdm(dataloader, desc="Extracting latents")
    batch_idx = 0
    n_samples = 0
    sum_ = 0
    sum_sq = 0
    count = 0

    with torch.no_grad():
        for waveform, _, _, _ in pbar:
            waveform = waveform.to(device)
            stft_spec = stft_transform(waveform)
            log_mag, phase = compute_magnitude_and_phase(stft_spec)
            x = torch.cat([log_mag, phase], dim=1).to(device)

            if variational:
                feat, mu, logvar = model.encoder(x)
                z = mu
            else:
                z = model.encoder(x)

            z = z.to(torch.float16).cpu()
            save_q.put((z, os.path.join(save_dir, f"batch_{batch_idx:05d}.pt")))

            sum_ += z.sum().item()
            sum_sq += (z.double() ** 2).sum().item()
            count += z.numel()

            batch_idx += 1
            n_samples += z.shape[0]

    # esperar a que el hilo termine de volcar todo a disco
    save_q.put(None)
    saver_thread.join()
    
    mean = sum_ / count
    std = (sum_sq / count - mean ** 2) ** 0.5
    
    stats_path = os.path.join(save_dir, "stats.json")
    
    with open(stats_path, "w") as f:
        json.dump({"mean": mean, "std": std, "n_samples": n_samples}, f)

    return n_samples, sum_, sum_sq, count

def extract_stats(save_dir):
    ''' No es necesario ejecutar esto si ya se ha ejecutado extract_latents, porque ya guarda los stats en stats.json '''
    batch_files = sorted(
        os.path.join(save_dir, f) for f in os.listdir(save_dir)
        if f.startswith("batch_") and f.endswith(".pt")
    )

    sum_ = 0.0
    sum_sq = 0.0
    count = 0
    n_samples = 0

    for fname in tqdm(batch_files, desc="Computing stats"):
        z = torch.load(fname, map_location='cpu').double()  # subir a double para evitar overflow/error de precision
        sum_ += z.sum().item()
        sum_sq += (z ** 2).sum().item()
        count += z.numel()
        n_samples += z.shape[0]

    mean = sum_ / count
    std = (sum_sq / count - mean ** 2) ** 0.5

    stats_path = os.path.join(save_dir, "stats.json")
    with open(stats_path, "w") as f:
        json.dump({"mean": mean, "std": std, "n_samples": n_samples}, f)

    print(f"Stats guardadas en {stats_path}: mean={mean:.4f}, std={std:.4f}, n_samples={n_samples}")
    return mean, std

def normalize_latents(save_dir):
    stats_path = os.path.join(save_dir, "stats.json")
    with open(stats_path, "r") as f:
        stats = json.load(f)
    mean = stats["mean"]
    std = stats["std"]

    batch_files = sorted(
        os.path.join(save_dir, f) for f in os.listdir(save_dir)
        if f.startswith("batch_") and f.endswith(".pt")
    )

    for fname in tqdm(batch_files, desc="Normalizing batches"):
        z = torch.load(fname, map_location='cpu')
        z = (z - mean) / std
        torch.save(z, fname)

    stats["normalized"] = False
    with open(stats_path, "w") as f:
        json.dump(stats, f)

    print(f"Normalizados {len(batch_files)} batches en {save_dir}")

In [ ]:
def conditions_to_device(conditions, device):
    return {k: v.to(device) for k, v in conditions.items()}


def extract_latents_cvae(model, dataset, save_dir, variational=True, batch_size=256, save_queue_size=8):
    """
    Extrae los latentes del cVAE (z = mu, determinista) para cada muestra del dataset.
    El dataset ya devuelve el mel precalculado (no hay waveform ni STFT que aplicar aqui).
    """
    os.makedirs(save_dir, exist_ok=True)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True,
        collate_fn=nsynth_collate_fn,
    )

    save_q = queue.Queue(maxsize=save_queue_size)

    def saver_worker():
        while True:
            item = save_q.get()
            if item is None:
                save_q.task_done()
                break
            tensor, path = item
            torch.save(tensor, path)
            save_q.task_done()

    saver_thread = threading.Thread(target=saver_worker, daemon=True)
    saver_thread.start()

    model.eval()

    reparam_original = model.reparameterize
    if variational:
        model.reparameterize = lambda mu, logvar: mu

    pbar = tqdm(dataloader, desc="Extracting latents (cVAE)")
    batch_idx = 0
    n_samples = 0
    sum_ = 0
    sum_sq = 0
    count = 0

    try:
        with torch.no_grad():
            for mels, keys, metadatas, features, conditions in pbar:
                mel = mels.to(device)
                cond = conditions_to_device(conditions, device)

                mel_hat, ddsp_params, kld, mu, logvar = model(
                    mel,
                    cond['instrument_onehot'],
                    cond['pitch_norm'],
                    cond['velocity_norm'],
                    cond['brightness'],
                    cond['sustain'],
                )

                z = mu.to(torch.float16).cpu()
                save_q.put((z, os.path.join(save_dir, f"batch_{batch_idx:05d}.pt")))

                sum_ += z.sum().item()
                sum_sq += (z.double() ** 2).sum().item()
                count += z.numel()

                batch_idx += 1
                n_samples += z.shape[0]
    finally:
        model.reparameterize = reparam_original
        save_q.put(None)
        saver_thread.join()

    mean = sum_ / count
    std = (sum_sq / count - mean ** 2) ** 0.5

    stats_path = os.path.join(save_dir, "stats.json")
    with open(stats_path, "w") as f:
        json.dump({"mean": mean, "std": std, "n_samples": n_samples}, f)

    print(f"Guardados {batch_idx} batches ({n_samples} muestras) en {save_dir}")
    return n_samples, sum_, sum_sq, count

## VAE

In [ ]:
modes = ['validation', 'training']
model_str = 'vae_diffusion' 

model = load_VAE(device=device)

n_fft = 1500
hop_length = 250
win_length = n_fft

stft_transform = T.Spectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length,
    power=None, onesided=True, center=False
).to(device)

for mode in modes:
    print(f"Processing {mode} set...")
    dataset = NSynth(mode)
    extract_latents_VAE(model, dataset, stft_transform, save_dir=PATHS[model_str][f'dataset_{mode}'], variational=variational)
    extract_stats(save_dir=PATHS[model_str][f'dataset_{mode}'], )
    normalize_latents(save_dir=PATHS[model_str][f'dataset_{mode}'])

## CVAE

In [ ]:
modes = ['validation', 'training']
model_str = 'cvae_diffusion'

model = load_cVAE(device=device)

for mode in modes:
    print(f"Processing {mode} set...")
    dataset = NSynth(mode)
    extract_latents_cvae(model, dataset, save_dir=PATHS[model_str][f'dataset_{mode}'], variational=True)
    extract_stats(save_dir=PATHS[model_str][f'dataset_{mode}'])
    normalize_latents(save_dir=PATHS[model_str][f'dataset_{mode}'])